# 工具
许多 AI 应用程序通过自然语言与用户交互。然而，某些用例需要模型使用结构化输入直接与外部系统（例如 API、数据库或文件系统）交互。在这些场景中，工具调用使模型能够生成符合指定输入架构的请求。

工具封装了可调用函数及其输入模式。这些可以传递给兼容的聊天模型，让模型决定是否调用工具以及使用哪些参数。

## 工具调用

<img src="https://langchain-ai.github.io/langgraph/concepts/img/tool_call.png">

工具调用通常是有条件的。根据用户输入和可用工具，模型可能会选择发出工具调用请求。此请求以`AIMessage`对象的形式返回，该对象包含一个`tool_calls`指定工具名称和输入参数的字段：

In [ ]:
llm_with_tools.invoke("What is 2 multiplied by 3?")
# -> AIMessage(tool_calls=[{'name': 'multiply', 'args': {'a': 2, 'b': 3}, ...}])

```shell

AIMessage(
  tool_calls=[
    ToolCall(name="multiply", args={"a": 2, "b": 3}),
    ...
  ]
)
```

重要的是，模型本身并不执行工具，它只生成请求。一个单独的执行器（例如运行时或代理）负责处理工具调用并返回结果。

更多详细信息请参阅工具调用指南。



## 预建工具¶
LangChain 为常见的外部系统（包括 API、数据库、文件系统和 Web 数据）提供预构建的工具集成。

浏览集成目录以查找可用的工具。

常见类别：

- 搜索：Bing、SerpAPI、Tavily
- 代码执行：Python REPL、Node.js REPL
- 数据库：SQL、MongoDB、Redis
- Web 数据：抓取和浏览
- API：OpenWeatherMap、NewsAPI等。

## 自定义工具¶
你可以使用装饰器或普通 Python 函数来定义自定义工具@tool。例如：

API 参考：工具

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

## 工具执行¶
虽然模型决定何时调用工具，但工具调用的执行必须由运行时组件处理。

LangGraph 为此提供了预构建的组件：

- `ToolNode`：执行工具的预建节点。
- `create_react_agent`：构建一个自动管理工具调用的完整代理。

# 执行工具¶
工具封装了可调用函数及其输入模式。这些可以传递给兼容的聊天模型，让模型决定是否调用工具并确定合适的参数。

您可以定义自己的工具或使用预建的工具



## 定义工具¶
使用@tool装饰器定义一个基本工具：

API 参考：工具

In [1]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

## 运行工具¶
工具符合Runnable 接口，这意味着您可以使用该方法运行工具invoke：


In [2]:
multiply.invoke({"a": 6, "b": 7})  # returns 42

42

如果使用 type="tool_call" 调用该工具，它将返回一个ToolMessage：

In [3]:
tool_call = {
    "type": "tool_call",
    "id": "1",
    "args": {"a": 42, "b": 7}
}
multiply.invoke(tool_call) # returns a ToolMessage object

ToolMessage(content='294', name='multiply', tool_call_id='1')

## 在Agent中使用¶
要创建工具调用代理，您可以使用预先构建的`create_react_agent`：

API 参考：[工具](https://python.langchain.com/api_reference/core/tools/langchain_core.tools.convert.tool.html?_gl=1*1ywibux*_gcl_au*NDE1NjY4Mjc0LjE3NTM0Mjc3MTc.*_ga*MTEyMjY1OTA5MS4xNzUzNDI3NzE4*_ga_47WX3HKKY2*czE3NTM5MjIwMDQkbzIyJGcwJHQxNzUzOTIyMDA0JGo2MCRsMCRoMA..)| [create_react_agent](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.chat_agent_executor.create_react_agent)

In [5]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
load_dotenv()

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[multiply]
)
agent.invoke({"messages": [{"role": "user", "content": "what's 42 x 7?"}]})

{'messages': [HumanMessage(content="what's 42 x 7?", additional_kwargs={}, response_metadata={}, id='aca48def-533b-47c4-ad21-e7190e6070f8'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_wpfCyR3XoqXS5K0eiS87wXId', 'function': {'arguments': '{"a":42,"b":7}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzBedTM5Yhhy5zMtdchgILNZv69FF', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2e5e4bbb-50c7-4663-a352-588e8819ae53-0', tool_calls=[{'name': 'multiply', 'args': {'a': 42, 'b': 7}, 'id': 'call_wpfCyR3XoqXS5K0eiS87wXId', 'type': 'tool_call'}], usage_metad

## 在工作流中使用¶
如果您正在编写自定义工作流程，则需要：

- 使用聊天模型注册工具
- 如果模型决定使用该工具，则调用该工具

用于`model.bind_tools()`将工具注册到模型中。

API 参考：init_chat_model

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-3-5-haiku-latest")

model_with_tools = model.bind_tools([multiply])

LLM 会自动确定是否需要调用工具，并使用适当的参数来调用该工具。

### 工具节点¶
要在自定义工作流中执行工具，请使用预构建的ToolNode或实现您自己的自定义节点。

`ToolNode`是用于在工作流中执行工具的专用节点。它提供以下功能：

- 支持同步和异步工具。
- 同时执行多个工具。
- 处理工具执行期间的错误（handle_tool_errors=True，默认启用）。有关更多详细信息，请参阅[处理工具错误](https://langchain-ai.github.io/langgraph/how-tos/tool-calling/#handle-errors)。

`ToolNode` 对 `MessagesState` 进行操作：

- 输入：`MessagesState`，其中最后一条消息是包含`tool_calls`参数的`AIMessage`。

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_wpfCyR3XoqXS5K0eiS87wXId', 'function': {'arguments': '{"a":42,"b":7}', 'name': 'multiply'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzBedTM5Yhhy5zMtdchgILNZv69FF', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2e5e4bbb-50c7-4663-a352-588e8819ae53-0', tool_calls=[{'name': 'multiply', 'args': {'a': 42, 'b': 7}, 'id': 'call_wpfCyR3XoqXS5K0eiS87wXId', 'type': 'tool_call'}], usage_metadata={'input_tokens': 52, 'output_tokens': 17, 'total_tokens': 69, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

- 输出：通过执行工具的生成工具进行更新的`MessagessTate`。

ToolMessage(content='294', name='multiply', id='a0bf403f-894e-41a6-bcdc-ac94ead15d72', tool_call_id='call_wpfCyR3XoqXS5K0eiS87wXId')

In [ ]:
from langgraph.prebuilt import ToolNode

def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

def get_coolest_cities():
    """Get a list of coolest cities"""
    return "nyc, sf"

tool_node = ToolNode([get_weather, get_coolest_cities])
tool_node.invoke({"messages": [...]})

In [6]:
# Single tool call


from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

# Define tools
@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

tool_node = ToolNode([get_weather])

message_with_single_tool_call = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"location": "sf"},
            "id": "tool_call_id",
            "type": "tool_call",
        }
    ],
)

tool_node.invoke({"messages": [message_with_single_tool_call]})

{'messages': [ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='tool_call_id')]}

In [7]:
### Multiple tool calls

from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

# Define tools

def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

def get_coolest_cities():
    """Get a list of coolest cities"""
    return "nyc, sf"

tool_node = ToolNode([get_weather, get_coolest_cities])

message_with_multiple_tool_calls = AIMessage(
    content="",
    tool_calls=[
        {
            "name": "get_coolest_cities",
            "args": {},
            "id": "tool_call_id_1",
            "type": "tool_call",
        },
        {
            "name": "get_weather",
            "args": {"location": "sf"},
            "id": "tool_call_id_2",
            "type": "tool_call",
        },
    ],
)

tool_node.invoke({"messages": [message_with_multiple_tool_calls]})

{'messages': [ToolMessage(content='nyc, sf', name='get_coolest_cities', tool_call_id='tool_call_id_1'),
  ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='tool_call_id_2')]}

In [9]:
message_with_multiple_tool_calls

AIMessage(content='', additional_kwargs={}, response_metadata={}, tool_calls=[{'name': 'get_coolest_cities', 'args': {}, 'id': 'tool_call_id_1', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': 'sf'}, 'id': 'tool_call_id_2', 'type': 'tool_call'}])

In [8]:
#### Use with a chat model

from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode

def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

tool_node = ToolNode([get_weather])

model = init_chat_model(model="gpt-4o-mini")
model_with_tools = model.bind_tools([get_weather])  


response_message = model_with_tools.invoke("what's the weather in sf?")
print(1, response_message)
tool_node.invoke({"messages": [response_message]})

1 content='' additional_kwargs={'tool_calls': [{'id': 'call_Wmo18RiX5OVgcqGaI7vGJryN', 'function': {'arguments': '{"location":"San Francisco"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 51, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzCBTxWXkdWY5VdzMo4tHmfE001YM', 'finish_reason': 'tool_calls', 'logprobs': None} id='run--890174ee-f987-483d-bb22-afda547a0cb4-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'San Francisco'}, 'id': 'call_Wmo18RiX5OVgcqGaI7vGJryN', 'type': 'tool_call'}] usage_metadata={'input_tokens': 51, 'output_tokens': 15, 'total_tokens': 66, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'out

{'messages': [ToolMessage(content="It's 60 degrees and foggy.", name='get_weather', tool_call_id='call_Wmo18RiX5OVgcqGaI7vGJryN')]}

In [ ]:
###Use in a tool-calling agent

from langchain.chat_models import init_chat_model
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState, START, END

def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    else:
        return "It's 90 degrees and sunny."

tool_node = ToolNode([get_weather])

model = init_chat_model(model="claude-3-5-haiku-latest")
model_with_tools = model.bind_tools([get_weather])

def should_continue(state: MessagesState):
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return END

def call_model(state: MessagesState):
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

builder = StateGraph(MessagesState)

# Define the two nodes we will cycle between
builder.add_node("call_model", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "call_model")
builder.add_conditional_edges("call_model", should_continue, ["tools", END])
builder.add_edge("tools", "call_model")

graph = builder.compile()

graph.invoke({"messages": [{"role": "user", "content": "what's the weather in sf?"}]})

## 工具定制

为了更好地控制工具行为，请使用`@tool`装饰器。

### 参数说明¶
从文档字符串自动生成描述：

API 参考：工具

In [ ]:
from langchain_core.tools import tool

@tool("multiply_tool", parse_docstring=True)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a: First operand
        b: Second operand
    """
    return a * b

### 显式输入模式¶
使用args_schema以下方式定义架构：

API 参考：工具

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class MultiplyInputSchema(BaseModel):
    """Multiply two numbers"""
    a: int = Field(description="First operand")
    b: int = Field(description="Second operand")

@tool("multiply_tool", args_schema=MultiplyInputSchema)
def multiply(a: int, b: int) -> int:
    return a * b

### 工具名称¶
使用第一个参数或名称属性覆盖默认工具名称：

API 参考：工具

In [ ]:
from langchain_core.tools import tool

@tool("multiply_tool")
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

## 下文管理¶
LangGraph 中的工具有时需要上下文数据，例如仅在运行时使用的参数（例如，用户 ID 或会话详细信息），这些数据不应由模型控制。LangGraph 提供了三种方法来管理此类上下文：

| 类型 | 使用场景 | 可变的 | 寿命 |
| :--- | :--- | :--- | :--- |
| 配置 | 静态、不可变的运行时数据 | X | 单次调用 |
| 短期记忆 | 调用期间动态变化的数据 |  | 单次调用 |
| 长期记忆 | 持久的跨会话数据 |  | 跨多个会话 |


### 配置¶
当您拥有工具所需的不可变运行时数据（例如用户标识符）时，请使用配置。您可以通过RunnableConfig调用时传递这些参数，并在工具中访问它们：

In [ ]:
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

@tool
def get_user_info(config: RunnableConfig) -> str:
    """Retrieve user information based on user ID."""
    user_id = config["configurable"].get("user_id")
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

# Invocation example with an agent
agent.invoke(
    {"messages": [{"role": "user", "content": "look up user info"}]},
    config={"configurable": {"user_id": "user_123"}}
)

扩展示例：在工具中访问配置

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

def get_user_info(
    config: RunnableConfig,
) -> str:
    """Look up user info."""
    user_id = config["configurable"].get("user_id")
    return "User is John Smith" if user_id == "user_123" else "Unknown user"

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[get_user_info],
)

agent.invoke(
    {"messages": [{"role": "user", "content": "look up user information"}]},
    config={"configurable": {"user_id": "user_123"}}
)

## 短期记忆¶
短期记忆保持在单次执行期间发生变化的动态状态。

要访问（读取）工具内的图形状态，可以使用特殊参数注释— InjectedState：

API 参考：工具| InjectedState | create_react_agent | AgentState

In [ ]:
from typing import Annotated, NotRequired
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState, create_react_agent
from langgraph.prebuilt.chat_agent_executor import AgentState

class CustomState(AgentState):
    # The user_name field in short-term state
    user_name: NotRequired[str]

@tool
def get_user_name(
    state: Annotated[CustomState, InjectedState]
) -> str:
    """Retrieve the current user-name from state."""
    # Return stored name or a default if not set
    return state.get("user_name", "Unknown user")

# Example agent setup
agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[get_user_name],
    state_schema=CustomState,
)

# Invocation: reads the name from state (initially empty)
agent.invoke({"messages": "what's my name?"})

使用Command返回更新user_name并附加确认消息的工具 ：

In [ ]:
from typing import Annotated
from langgraph.types import Command
from langchain_core.messages import ToolMessage
from langchain_core.tools import tool, InjectedToolCallId

@tool
def update_user_name(
    new_name: str,
    tool_call_id: Annotated[str, InjectedToolCallId]
) -> Command:
    """Update user-name in short-term memory."""
    return Command(update={
        "user_name": new_name,
        "messages": [
            ToolMessage(f"Updated user name to {new_name}", tool_call_id=tool_call_id)
        ]
    })

如果您想使用Command返回和更新图形状态的工具，您可以使用预构建`create_react_agent`/`ToolNode`组件，或者实现您自己的工具执行节点，收集`Command`工具返回的对象并返回它们的列表，例如：

In [ ]:
def call_tools(state):
    ...
    commands = [tools_by_name[tool_call["name"]].invoke(tool_call) for tool_call in tool_calls]
    return commands

## 长期记忆¶
使用长期记忆来存储对话中特定于用户或应用程序的数据。这对于像聊天机器人这样的应用程序非常有用，因为您需要记住用户的偏好或其他信息。

要使用长期记忆，您需要：

配置存储以在调用之间保留数据。
- 从工具内访问商店。
- 要访问商店中的信息：

API 参考：RunnableConfig |工具| StateGraph | get_store

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.graph import StateGraph
from langgraph.config import get_store

@tool
def get_user_info(config: RunnableConfig) -> str:
    """Look up user info."""
    # Same as that provided to `builder.compile(store=store)`
    # or `create_react_agent`
    store = get_store()
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

builder = StateGraph(...)
...
graph = builder.compile(store=store)

访问长期记忆

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.config import get_store
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore

store = InMemoryStore() 

store.put(  
    ("users",),  
    "user_123",  
    {
        "name": "John Smith",
        "language": "English",
    } 
)

@tool
def get_user_info(config: RunnableConfig) -> str:
    """Look up user info."""
    # Same as that provided to `create_react_agent`
    store = get_store() 
    user_id = config["configurable"].get("user_id")
    user_info = store.get(("users",), user_id) 
    return str(user_info.value) if user_info else "Unknown user"

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[get_user_info],
    store=store 
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "look up user information"}]},
    config={"configurable": {"user_id": "user_123"}}
)

In [ ]:
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langgraph.graph import StateGraph
from langgraph.config import get_store

@tool
def save_user_info(user_info: str, config: RunnableConfig) -> str:
    """Save user info."""
    # Same as that provided to `builder.compile(store=store)`
    # or `create_react_agent`
    store = get_store()
    user_id = config["configurable"].get("user_id")
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

builder = StateGraph(...)
...
graph = builder.compile(store=store)

In [ ]:
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langgraph.config import get_store
from langgraph.prebuilt import create_react_agent
from langgraph.store.memory import InMemoryStore

store = InMemoryStore() 

class UserInfo(TypedDict): 
    name: str

@tool
def save_user_info(user_info: UserInfo, config: RunnableConfig) -> str: 
    """Save user info."""
    # Same as that provided to `create_react_agent`
    store = get_store() 
    user_id = config["configurable"].get("user_id")
    store.put(("users",), user_id, user_info) 
    return "Successfully saved user info."

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[save_user_info],
    store=store
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "My name is John Smith"}]},
    config={"configurable": {"user_id": "user_123"}} 
)

# You can access the store directly to get the value
store.get(("users",), "user_123").value

## 高级工具功能¶
### 立即返回¶
用于`return_direct=True`立即返回工具的结果，而无需执行额外的逻辑。

这对于不应触发进一步处理或工具调用的工具很有用，允许您将结果直接返回给用户。

In [ ]:
@tool(return_direct=True)
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

In [10]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool(return_direct=True)
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[add]
)

agent.invoke(
    {"messages": [{"role": "user", "content": "what's 3 + 5?"}]}
)

{'messages': [HumanMessage(content="what's 3 + 5?", additional_kwargs={}, response_metadata={}, id='5ea3ab1f-d1bd-44e3-92b5-d9340d65b776'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_By4dPeEGiN1JDGJH9ixpS3Tz', 'function': {'arguments': '{"a":3,"b":5}', 'name': 'add'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-BzCg3j78USgDCIscqWhJbHaEsZuuY', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--7efac7b8-f2d4-4043-b18e-a78e7232015d-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_By4dPeEGiN1JDGJH9ixpS3Tz', 'type': 'tool_call'}], usage_metadata={'input_t

> 如果要构建自定义工作流，并且不依赖 create_react_agent 或 ToolNode，则还需要实现控制流，以处理 return_direct=True。

### 强制使用工具 
如果需要强制使用特定工具，则需要在模型级别使用 `bind_tools` 方法中的 `tool_choice` 参数进行配置。

通过 tool_choice 强制使用特定工具：

In [ ]:
@tool(return_direct=True)
def greet(user_name: str) -> int:
    """Greet user."""
    return f"Hello {user_name}!"

tools = [greet]

configured_model = model.bind_tools(
    tools,
    # Force the use of the 'greet' tool
    tool_choice={"type": "tool", "name": "greet"}
)

In [ ]:
from langchain_core.tools import tool

@tool(return_direct=True)
def greet(user_name: str) -> int:
    """Greet user."""
    return f"Hello {user_name}!"

tools = [greet]

agent = create_react_agent(
    model=model.bind_tools(tools, tool_choice={"type": "tool", "name": "greet"}),
    tools=tools
)

agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, I am Bob"}]}
)

> 在没有停止条件的情况下强制使用工具可能会造成无限循环。请使用以下安全措施之一：          
> 用 标记该工具以`return_direct=True`在执行后结束循环。        
> 设置`recursion_limit`限制执行步骤的数量。         

该tool_choice参数用于配置模型在决定调用工具时应使用的工具。当您希望确保始终针对特定任务调用特定工具，或者希望覆盖模型根据其内部逻辑选择工具的默认行为时，此功能非常有用。

请注意，并非所有型号都支持此功能，具体配置可能因您使用的型号而异。

### 禁用并行调用
对于受支持的提供程序，可以通过 `model.bind_tools() `方法设置 `parallel_tool_calls=False` 来禁用并行工具调用：

In [ ]:
model.bind_tools(
    tools,
    parallel_tool_calls=False
)

In [ ]:
from langchain.chat_models import init_chat_model

def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

model = init_chat_model("anthropic:claude-3-5-sonnet-latest", temperature=0)
tools = [add, multiply]
agent = create_react_agent(
    # disable parallel tool calls
    model=model.bind_tools(tools, parallel_tool_calls=False),
    tools=tools
)

agent.invoke(
    {"messages": [{"role": "user", "content": "what's 3 + 5 and 4 * 7?"}]}
)

### 处理错误¶
LangGraph 通过预构建的ToolNode组件为工具执行提供内置错误处理，可独立使用或在预构建代理中使用。

默认情况下，ToolNode捕获工具执行期间引发的异常并将其作为ToolMessage具有指示错误的状态的对象返回。

API 参考：AIMessage | ToolNode

In [13]:
from langchain_core.messages import AIMessage
from langgraph.prebuilt import ToolNode

def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together and returns the result."""
    if a == 42:
        raise ValueError("The ultimate error")
    return a * b

# Default error handling (enabled by default)
tool_node = ToolNode([multiply])

message = AIMessage(
    content="",
    tool_calls=[{
        "name": "multiply",
        "args": {"a": 42, "b": 7},
        "id": "tool_call_id",
        "type": "tool_call"
    }]
)

result = tool_node.invoke({"messages": [message]})
result

{'messages': [ToolMessage(content="Error: ValueError('The ultimate error')\n Please fix your mistakes.", name='multiply', tool_call_id='tool_call_id', status='error')]}

#### 禁用错误处理¶
要直接传播异常，请禁用错误处理：

In [ ]:
tool_node = ToolNode([multiply], handle_tool_errors=False)

#### 自定义错误消息¶
通过将错误处理参数设置为字符串来提供自定义错误消息：

In [ ]:
tool_node = ToolNode(
    [multiply],
    handle_tool_errors="Can't use 42 as the first operand, please switch operands!"
)

示例输出：


```shell

{'messages': [
    ToolMessage(
        content="Can't use 42 as the first operand, please switch operands!",
        name='multiply',
        tool_call_id='tool_call_id',
        status='error'
    )
]}
```


### 代理中的错误处理¶
预建代理中的错误处理（create_react_agent）利用ToolNode：

API 参考：create_react_agent

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=[multiply]
)

# Default error handling
agent.invoke({"messages": [{"role": "user", "content": "what's 42 x 7?"}]})

要在预建代理中禁用或自定义错误处理，请明确传递已配置的ToolNode：

In [ ]:
custom_tool_node = ToolNode(
    [multiply],
    handle_tool_errors="Cannot use 42 as a first operand!"
)

agent_custom = create_react_agent(
    model="anthropic:claude-3-7-sonnet-latest",
    tools=custom_tool_node
)

agent_custom.invoke({"messages": [{"role": "user", "content": "what's 42 x 7?"}]})

### 处理大量工具¶
随着可用工具数量的增加，您可能希望限制 LLM 的选择范围，以减少令牌消耗并帮助管理 LLM 推理中的错误源。

为了解决这个问题，您可以通过在运行时使用语义搜索检索相关工具来动态调整模型可用的工具。

请参阅[langgraph-bigtool](https://github.com/langchain-ai/langgraph-bigtool)预构建库以获得可立即使用的实现。

## 预建工具¶
LLM 提供者工具

通过向 `create_react_agent` 的 `tools` 参数传递包含工具规格的字典，可以使用模型提供者预制的工具。例如，使用 OpenAI 的 `web_search_preview` 工具：


In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model="openai:gpt-4o-mini",
    tools=[{"type": "web_search_preview"}]
)
response = agent.invoke(
    {"messages": ["What was a positive news story from today?"]}
)

请查阅您正在使用的特定型号的文档，了解可用的工具以及如何使用它们。

## LangChain 工具¶
此外，LangChain 支持各种预构建工具集成，可与 API、数据库、文件系统、Web 数据等进行交互。这些工具扩展了代理的功能并实现了快速开发。

您可以在LangChain 集成目录中浏览可用集成的完整列表。

一些常用的工具类别包括：

- 搜索：Bing、SerpAPI、Tavily
- 代码解释器：Python REPL、Node.js REPL
- 数据库：SQL、MongoDB、Redis
- Web 数据：Web 抓取和浏览
- API：OpenWeatherMap、NewsAPI 等

这些集成可以使用上述示例中的相同tools参数进行配置并添加到代理中。